Part 5A: Initialization

In [1]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ============================================================
# Notebook 5
# Downstream Classification Evaluation
# ============================================================

print("=" * 70)
print("Notebook 5 : Downstream Classification Evaluation")
print("=" * 70)

Notebook 5 : Downstream Classification Evaluation


In [3]:
# ============================================================
# Imports
# ============================================================

import os
import gc
import json
import time
import random
import warnings

from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Classifiers
# ------------------------------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

In [4]:
# ============================================================
# Load Configuration
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/TopicEvalBench"
)

CONFIG_FILE = PROJECT_DIR /"config/config.json"

with open(CONFIG_FILE, "r") as f:
    CONFIG = json.load(f)

N_FOLDS = CONFIG["n_folds"]
SEED = CONFIG["random_seed"]

print("Project Name :", CONFIG["project_name"])
print("Number of Folds :", N_FOLDS)
print("Random Seed :", SEED)

Project Name : TopicEvalBench
Number of Folds : 5
Random Seed : 42


In [5]:
# ============================================================
# Reproducibility
# ============================================================

random.seed(SEED)

np.random.seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)

print("Random Seed =", SEED)

Random Seed = 42


In [6]:
# ============================================================
# Load Experiment Manifest
# ============================================================

MODEL_DIR = PROJECT_DIR /"models"
manifest_file = MODEL_DIR / "experiment_manifest_stage4.csv"

manifest = pd.read_csv(manifest_file)

print("Manifest Loaded Successfully")
print("Total Experiments :", len(manifest))
print()

display(manifest.head())

Manifest Loaded Successfully
Total Experiments : 495



,ExperimentID,Fold,Topics,Status,TrainingDocuments,DictionarySize,RandomSeed,Passes,Iterations,Alpha,...,LogPerplexityTrain_y,Perplexity,UMass,CV,UCI,NPMI,LogPerplexityTest,PerplexityTest,DeltaLogPerplexity,RPC
0,F1_K002,1,2,Available,240,891,42,20,400,"[0.5, 0.5]",...,-6.298535,78.713294,-4.680316,0.308302,-4.338849,-0.128971,-6.612982,97.908976,NaN,NaN
1,F1_K003,1,3,Available,240,891,42,20,400,"[0.3333333432674408, 0.3333333432674408, 0.333...",...,-6.214306,74.249327,-3.103219,0.382957,-2.588816,-0.045408,-6.737341,106.757182,-0.124359,0.124359
2,F1_K004,1,4,Available,240,891,42,20,400,"[0.25, 0.25, 0.25, 0.25]",...,-6.193754,73.199113,-2.827298,0.442629,-2.979363,-0.055813,-6.864329,116.595454,-0.126987,0.126987
3,F1_K005,1,5,Available,240,891,42,20,400,"[0.20000000298023224, 0.20000000298023224, 0.2...",...,-6.191917,73.105977,-2.914309,0.430353,-2.637528,-0.026696,-7.023668,130.243657,-0.159339,0.159339
4,F1_K006,1,6,Available,240,891,42,20,400,"[0.1666666716337204, 0.1666666716337204, 0.166...",...,-6.224693,74.785806,-2.840108,0.462751,-2.580816,-0.021140,-7.153422,142.448092,-0.129754,0.129754


In [7]:
# ============================================================
# Create Output Directories
# ============================================================

CLASSIFICATION_DIR = PROJECT_DIR / "classification"

PREDICTION_DIR = CLASSIFICATION_DIR / "Predictions"

CLASSIFICATION_DIR.mkdir(exist_ok=True)

PREDICTION_DIR.mkdir(exist_ok=True)

for fold in range(1, N_FOLDS + 1):

    (PREDICTION_DIR / f"Fold_{fold}").mkdir(exist_ok=True)

print("Output folders created.")

Output folders created.


In [8]:
# ============================================================
# Evaluation Metrics
# ============================================================

def evaluate_predictions(y_true, y_pred):

    results = {

        "Accuracy":
            accuracy_score(y_true, y_pred),

        "BalancedAccuracy":
            balanced_accuracy_score(y_true, y_pred),

        "PrecisionMacro":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "RecallMacro":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "F1Macro":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "MCC":
            matthews_corrcoef(
                y_true,
                y_pred
            ),

        "Kappa":
            cohen_kappa_score(
                y_true,
                y_pred
            )

    }

    return results

In [9]:
# ============================================================
# Notebook Summary
# ============================================================

print("=" * 70)
print("Notebook 5 Initialization Complete")
print("=" * 70)

print(f"Experiments : {len(manifest)}")
print(f"Folds       : {N_FOLDS}")

print()
print("Classifiers")
print("  1. Logistic Regression")
print("  2. Random Forest")

print()
print("Metrics")
print("  Accuracy")
print("  Balanced Accuracy")
print("  Precision (Macro)")
print("  Recall (Macro)")
print("  F1 (Macro)")
print("  MCC")
print("  Cohen's Kappa")

print("=" * 70)

Notebook 5 Initialization Complete
Experiments : 495
Folds       : 5

Classifiers
  1. Logistic Regression
  2. Random Forest

Metrics
  Accuracy
  Balanced Accuracy
  Precision (Macro)
  Recall (Macro)
  F1 (Macro)
  MCC
  Cohen's Kappa


Part 5B: Load & Validate θ Features

In [10]:

def load_theta_dataset(experiment):

    # --------------------------------------------------------
    # Theta Features
    # --------------------------------------------------------

    train_theta = pd.read_csv(experiment["ThetaTrainFile"])
    test_theta  = pd.read_csv(experiment["ThetaTestFile"])

    # Keep only topic probabilities
    X_train = train_theta.filter(regex=r"^Topic_")
    X_test  = test_theta.filter(regex=r"^Topic_")

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    y_train = train_theta["LabelID"].to_numpy()
    y_test  = test_theta["LabelID"].to_numpy()

    return X_train, X_test, y_train, y_test

In [11]:
# ============================================================
# Validate Dataset
# ============================================================

def validate_dataset(
    experiment,
    X_train,
    X_test,
    y_train,
    y_test
):
    """
    Validate one experiment before classification.
    """

    expected_topics = int(experiment["Topics"])

    # --------------------------------------------------------
    # Sample Counts
    # --------------------------------------------------------

    assert X_train.shape[0] == len(y_train), \
        "Training sample mismatch."

    assert X_test.shape[0] == len(y_test), \
        "Testing sample mismatch."

    # --------------------------------------------------------
    # Number of Features
    # --------------------------------------------------------

    assert X_train.shape[1] == expected_topics, \
        f"Expected {expected_topics} training features, got {X_train.shape[1]}."

    assert X_test.shape[1] == expected_topics, \
        f"Expected {expected_topics} testing features, got {X_test.shape[1]}."

    # --------------------------------------------------------
    # Missing Values
    # --------------------------------------------------------

    assert not X_train.isnull().values.any(), \
        "NaN found in ThetaTrain."

    assert not X_test.isnull().values.any(), \
        "NaN found in ThetaTest."

    # --------------------------------------------------------
    # Infinite Values
    # --------------------------------------------------------

    assert np.isfinite(X_train.values).all(), \
        "Infinite values in ThetaTrain."

    assert np.isfinite(X_test.values).all(), \
        "Infinite values in ThetaTest."

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    assert len(np.unique(y_train)) >= 2, \
        "Training labels contain only one class."

    assert len(np.unique(y_test)) >= 2, \
        "Testing labels contain only one class."

    return True

In [12]:
# ============================================================
# Display Dataset Information
# ============================================================

def dataset_summary(
    experiment,
    X_train,
    X_test,
    y_train,
    y_test
):

    print("=" * 60)

    print(f"Experiment : {experiment['ExperimentID']}")
    print(f"Fold       : {experiment['Fold']}")
    print(f"Topics     : {experiment['Topics']}")

    print("-" * 60)

    print("Training")

    print("Samples :", X_train.shape[0])
    print("Features:", X_train.shape[1])
    print("Classes :", len(np.unique(y_train)))

    print()

    print("Testing")

    print("Samples :", X_test.shape[0])
    print("Features:", X_test.shape[1])
    print("Classes :", len(np.unique(y_test)))

    print("=" * 60)

In [13]:
# ============================================================
# Test Dataset Loader
# ============================================================

experiment = manifest.iloc[0]

X_train, X_test, y_train, y_test = load_theta_dataset(experiment)

validate_dataset(
    experiment,
    X_train,
    X_test,
    y_train,
    y_test
)

dataset_summary(
    experiment,
    X_train,
    X_test,
    y_train,
    y_test
)

display(X_train.head())

Experiment : F1_K002
Fold       : 1
Topics     : 2
------------------------------------------------------------
Training
Samples : 240
Features: 2
Classes : 10

Testing
Samples : 60
Features: 2
Classes : 10


,Topic_1,Topic_2
0,0.613961,0.386039
1,0.014010,0.985990
2,0.007627,0.992373
3,0.011832,0.988168
4,0.011737,0.988263


In [14]:
# ============================================================
# Validate All Experiments
# ============================================================

print("=" * 70)
print("Validating All Experiments")
print("=" * 70)

for _, experiment in manifest.iterrows():

    X_train, X_test, y_train, y_test = load_theta_dataset(experiment)

    validate_dataset(
        experiment,
        X_train,
        X_test,
        y_train,
        y_test
    )

print(f"✓ Successfully validated {len(manifest)} experiments.")

Validating All Experiments
✓ Successfully validated 495 experiments.


Part 5C

In [15]:

# ============================================================
# Logistic Regression
# ============================================================

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    matthews_corrcoef,
    balanced_accuracy_score,
    classification_report
)

In [16]:

# ============================================================
# Evaluate Logistic Regression
# ============================================================

def evaluate_logistic_regression(
    experiment,
    random_state=SEED
):
    """
    Train and evaluate Logistic Regression for one experiment.
    """

    X_train, X_test, y_train, y_test = load_theta_dataset(experiment)

    clf = LogisticRegression(
        random_state=SEED,
        max_iter=1000,
        multi_class="auto"
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    balanced_acc = balanced_accuracy_score(
        y_test,
        y_pred
    )

    return {

        "ExperimentID": experiment["ExperimentID"],
        "Fold": experiment["Fold"],
        "Topics": experiment["Topics"],

        "Accuracy": accuracy,
        "BalancedAccuracy": balanced_acc,

        "Precision": precision,
        "Recall": recall,
        "F1": f1,

        "MCC": mcc
    }

In [17]:

# ============================================================
# Evaluate All Logistic Regression Models
# ============================================================

results = []

for _, experiment in manifest.iterrows():

    result = evaluate_logistic_regression(experiment)

    results.append(result)

logreg_results = pd.DataFrame(results)

display(logreg_results.head())

,ExperimentID,Fold,Topics,Accuracy,BalancedAccuracy,Precision,Recall,F1,MCC
0,F1_K002,1,2,0.333333,0.333333,0.285746,0.333333,0.234429,0.280900
1,F1_K003,1,3,0.483333,0.483333,0.535387,0.483333,0.467369,0.435578
2,F1_K004,1,4,0.666667,0.666667,0.686349,0.666667,0.657855,0.633355
3,F1_K005,1,5,0.566667,0.566667,0.582197,0.566667,0.560508,0.522076
4,F1_K006,1,6,0.750000,0.750000,0.756905,0.750000,0.746630,0.723788


In [18]:

# ============================================================
# Save Logistic Regression Results
# ============================================================

output_file = CLASSIFICATION_DIR / "logistic_regression_results.csv"

logreg_results.to_csv(
    output_file,
    index=False
)

print(f"Saved:\n{output_file}")

Saved:
/content/drive/MyDrive/TopicEvalBench/classification/logistic_regression_results.csv


In [21]:
# ============================================================
# Mean Performance Across Folds
# ============================================================

summary = (
    logreg_results
    .groupby("Topics", as_index=False)
    .agg({
        "Accuracy": "mean",
        "BalancedAccuracy": "mean",
        "Precision": "mean",
        "Recall": "mean",
        "F1": "mean",
        "MCC": "mean"
    })
)

display(summary.head())

,Topics,Accuracy,BalancedAccuracy,Precision,Recall,F1,MCC
0,2,0.330000,0.330000,0.320721,0.330000,0.259358,0.275742
1,3,0.416667,0.416667,0.398222,0.416667,0.382724,0.359682
2,4,0.540000,0.540000,0.521733,0.540000,0.510531,0.495699
3,5,0.583333,0.583333,0.578946,0.583333,0.561611,0.542811
4,6,0.676667,0.676667,0.688535,0.676667,0.662485,0.645581


In [22]:

# ============================================================
# Save Mean Performance
# ============================================================

summary_file = (
    CLASSIFICATION_DIR /
    "logistic_regression_summary.csv"
)

summary.to_csv(
    summary_file,
    index=False
)

print(summary_file)

/content/drive/MyDrive/TopicEvalBench/classification/logistic_regression_summary.csv


Part 5D = Random Forest classification

In [23]:
# ============================================================
# 5D-1 Random Forest Imports
# ============================================================

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    matthews_corrcoef,
    balanced_accuracy_score
)

In [24]:
# ============================================================
# 5D-2 Random Forest Configuration
# ============================================================

RF_N_ESTIMATORS = 500
RF_MAX_DEPTH = None
RF_MIN_SAMPLES_SPLIT = 2
RF_MIN_SAMPLES_LEAF = 1
RF_MAX_FEATURES = "sqrt"
RF_CLASS_WEIGHT = "balanced"
RF_N_JOBS = -1

print("Random Forest Configuration")
print("n_estimators      :", RF_N_ESTIMATORS)
print("max_depth         :", RF_MAX_DEPTH)
print("min_samples_split :", RF_MIN_SAMPLES_SPLIT)
print("min_samples_leaf  :", RF_MIN_SAMPLES_LEAF)
print("max_features      :", RF_MAX_FEATURES)
print("class_weight      :", RF_CLASS_WEIGHT)

Random Forest Configuration
n_estimators      : 500
max_depth         : None
min_samples_split : 2
min_samples_leaf  : 1
max_features      : sqrt
class_weight      : balanced


In [25]:
# ============================================================
# 5D-3 Evaluate One Experiment
# ============================================================

def evaluate_random_forest(
    experiment,
    random_state=SEED
):
    """
    Train and evaluate Random Forest using LDA theta features.
    """

    X_train, X_test, y_train, y_test = load_theta_dataset(
        experiment
    )

    clf = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_split=RF_MIN_SAMPLES_SPLIT,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        max_features=RF_MAX_FEATURES,
        class_weight=RF_CLASS_WEIGHT,
        random_state=random_state,
        n_jobs=RF_N_JOBS
    )

    clf.fit(
        X_train,
        y_train
    )

    y_pred = clf.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        y_pred
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    return {

        "ExperimentID": experiment["ExperimentID"],

        "Fold": experiment["Fold"],

        "Topics": experiment["Topics"],

        "Accuracy": accuracy,

        "BalancedAccuracy": balanced_accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc
    }

In [27]:
from tqdm.auto import tqdm

# ============================================================
# 5D-4 Evaluate All Experiments
# ============================================================

rf_results = []

print("=" * 70)
print("Random Forest Evaluation")
print("=" * 70)

for _, experiment in tqdm(
    manifest.iterrows(),
    total=len(manifest),
    desc="Random Forest"
):

    try:

        result = evaluate_random_forest(
            experiment
        )

        rf_results.append(
            result
        )

    except Exception as e:

        print(
            f"\nError: "
            f"Fold={experiment['Fold']}, "
            f"K={experiment['Topics']}"
        )

        print(e)

rf_results = pd.DataFrame(
    rf_results
)

display(
    rf_results.head()
)

print(
    f"\nSuccessfully evaluated "
    f"{len(rf_results)} experiments."
)


Random Forest Evaluation


Random Forest:   0%|          | 0/495 [00:00<?, ?it/s]

,ExperimentID,Fold,Topics,Accuracy,BalancedAccuracy,Precision,Recall,F1,MCC
0,F1_K002,1,2,0.233333,0.233333,0.297381,0.233333,0.220514,0.151997
1,F1_K003,1,3,0.466667,0.466667,0.523611,0.466667,0.460682,0.411881
2,F1_K004,1,4,0.533333,0.533333,0.615530,0.533333,0.533408,0.487538
3,F1_K005,1,5,0.566667,0.566667,0.608463,0.566667,0.567667,0.522566
4,F1_K006,1,6,0.633333,0.633333,0.635844,0.633333,0.618534,0.597406



Successfully evaluated 495 experiments.


In [28]:
# ============================================================
# 5D-5 Save Random Forest Results
# ============================================================

rf_results_file = (
    CLASSIFICATION_DIR /
    "random_forest_results.csv"
)

rf_results.to_csv(
    rf_results_file,
    index=False
)

print(
    f"Saved:\n{rf_results_file}"
)

Saved:
/content/drive/MyDrive/TopicEvalBench/classification/random_forest_results.csv


In [29]:
# ============================================================
# 5D-6 Mean Performance Across Folds
# ============================================================

rf_summary = (
    rf_results
    .groupby(
        "Topics",
        as_index=False
    )
    .agg({
        "Accuracy": "mean",
        "BalancedAccuracy": "mean",
        "Precision": "mean",
        "Recall": "mean",
        "F1": "mean",
        "MCC": "mean"
    })
)

display(
    rf_summary.head()
)

,Topics,Accuracy,BalancedAccuracy,Precision,Recall,F1,MCC
0,2,0.223333,0.223333,0.263849,0.223333,0.220721,0.139180
1,3,0.330000,0.330000,0.368304,0.330000,0.319071,0.258708
2,4,0.483333,0.483333,0.533994,0.483333,0.471446,0.432195
3,5,0.526667,0.526667,0.573405,0.526667,0.508367,0.481934
4,6,0.593333,0.593333,0.599748,0.593333,0.583732,0.551564


In [30]:
# ============================================================
# 5D-7 Standard Deviation Across Folds
# ============================================================

rf_std = (
    rf_results
    .groupby(
        "Topics",
        as_index=False
    )
    .agg({
        "Accuracy": "std",
        "BalancedAccuracy": "std",
        "Precision": "std",
        "Recall": "std",
        "F1": "std",
        "MCC": "std"
    })
)

rf_std = rf_std.rename(
    columns={
        "Accuracy": "AccuracyStd",
        "BalancedAccuracy": "BalancedAccuracyStd",
        "Precision": "PrecisionStd",
        "Recall": "RecallStd",
        "F1": "F1Std",
        "MCC": "MCCStd"
    }
)

display(
    rf_std.head()
)

,Topics,AccuracyStd,BalancedAccuracyStd,PrecisionStd,RecallStd,F1Std,MCCStd
0,2,0.088663,0.088663,0.110697,0.088663,0.085664,0.100080
1,3,0.093838,0.093838,0.100846,0.093838,0.096415,0.104627
2,4,0.065617,0.065617,0.084609,0.065617,0.071389,0.073477
3,5,0.038370,0.038370,0.053935,0.038370,0.046869,0.041802
4,6,0.050827,0.050827,0.046514,0.050827,0.050130,0.055948


In [31]:
# ============================================================
# 5D-8 Combined Random Forest Summary
# ============================================================

rf_summary = rf_summary.merge(
    rf_std,
    on="Topics",
    how="left"
)

display(
    rf_summary.head()
)

,Topics,Accuracy,BalancedAccuracy,Precision,Recall,F1,MCC,AccuracyStd,BalancedAccuracyStd,PrecisionStd,RecallStd,F1Std,MCCStd
0,2,0.223333,0.223333,0.263849,0.223333,0.220721,0.139180,0.088663,0.088663,0.110697,0.088663,0.085664,0.100080
1,3,0.330000,0.330000,0.368304,0.330000,0.319071,0.258708,0.093838,0.093838,0.100846,0.093838,0.096415,0.104627
2,4,0.483333,0.483333,0.533994,0.483333,0.471446,0.432195,0.065617,0.065617,0.084609,0.065617,0.071389,0.073477
3,5,0.526667,0.526667,0.573405,0.526667,0.508367,0.481934,0.038370,0.038370,0.053935,0.038370,0.046869,0.041802
4,6,0.593333,0.593333,0.599748,0.593333,0.583732,0.551564,0.050827,0.050827,0.046514,0.050827,0.050130,0.055948


In [32]:
# ============================================================
# 5D-9 Save Random Forest Summary
# ============================================================

rf_summary_file = (
    CLASSIFICATION_DIR /
    "random_forest_summary.csv"
)

rf_summary.to_csv(
    rf_summary_file,
    index=False
)

print(
    f"Saved:\n{rf_summary_file}"
)

Saved:
/content/drive/MyDrive/TopicEvalBench/classification/random_forest_summary.csv


Part 5E — Support Vector Machine

In [33]:
# ============================================================
# 5E-1 SVM Imports
# ============================================================

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    matthews_corrcoef,
    balanced_accuracy_score
)

In [34]:
# ============================================================
# 5E-2 SVM Configuration
# ============================================================

SVM_KERNEL = "rbf"
SVM_C = 1.0
SVM_GAMMA = "scale"
SVM_CLASS_WEIGHT = "balanced"

print("SVM Configuration")
print("kernel      :", SVM_KERNEL)
print("C           :", SVM_C)
print("gamma       :", SVM_GAMMA)
print("class_weight:", SVM_CLASS_WEIGHT)

SVM Configuration
kernel      : rbf
C           : 1.0
gamma       : scale
class_weight: balanced


In [36]:
# ============================================================
# 5E-3 Evaluate One Experiment
# ============================================================

def evaluate_svm(
    experiment,
    random_state=SEED
):
    """
    Train and evaluate SVM using LDA theta features.
    """

    X_train, X_test, y_train, y_test = load_theta_dataset(
        experiment
    )

    clf = SVC(
        kernel=SVM_KERNEL,
        C=SVM_C,
        gamma=SVM_GAMMA,
        class_weight=SVM_CLASS_WEIGHT,
        random_state=random_state
    )

    clf.fit(
        X_train,
        y_train
    )

    y_pred = clf.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        y_pred
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    return {

        "ExperimentID": experiment["ExperimentID"],

        "Fold": experiment["Fold"],

        "Topics": experiment["Topics"],

        "Accuracy": accuracy,

        "BalancedAccuracy": balanced_accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc
    }

In [37]:
# ============================================================
# 5E-4 Evaluate All SVM Experiments
# ============================================================

svm_results = []

print("=" * 70)
print("SVM Evaluation")
print("=" * 70)

for _, experiment in tqdm(
    manifest.iterrows(),
    total=len(manifest),
    desc="SVM"
):

    try:

        result = evaluate_svm(
            experiment
        )

        svm_results.append(
            result
        )

    except Exception as e:

        print(
            f"\nError: "
            f"Fold={experiment['Fold']}, "
            f"K={experiment['Topics']}"
        )

        print(e)

svm_results = pd.DataFrame(
    svm_results
)

display(
    svm_results.head()
)

print(
    f"\nSuccessfully evaluated "
    f"{len(svm_results)} experiments."
)

SVM Evaluation


SVM:   0%|          | 0/495 [00:00<?, ?it/s]

,ExperimentID,Fold,Topics,Accuracy,BalancedAccuracy,Precision,Recall,F1,MCC
0,F1_K002,1,2,0.316667,0.316667,0.232412,0.316667,0.205582,0.264306
1,F1_K003,1,3,0.516667,0.516667,0.502302,0.516667,0.457729,0.478107
2,F1_K004,1,4,0.600000,0.600000,0.582677,0.600000,0.557936,0.565414
3,F1_K005,1,5,0.650000,0.650000,0.730952,0.650000,0.623535,0.623552
4,F1_K006,1,6,0.700000,0.700000,0.786508,0.700000,0.682472,0.676553



Successfully evaluated 495 experiments.


In [38]:
# ============================================================
# 5E-5 Save SVM Results
# ============================================================

svm_results_file = (
    CLASSIFICATION_DIR /
    "svm_results.csv"
)

svm_results.to_csv(
    svm_results_file,
    index=False
)

print(
    f"Saved:\n{svm_results_file}"
)

Saved:
/content/drive/MyDrive/TopicEvalBench/classification/svm_results.csv


In [39]:
# ============================================================
# 5E-6 Mean Performance Across Folds
# ============================================================

svm_summary = (
    svm_results
    .groupby(
        "Topics",
        as_index=False
    )
    .agg({
        "Accuracy": "mean",
        "BalancedAccuracy": "mean",
        "Precision": "mean",
        "Recall": "mean",
        "F1": "mean",
        "MCC": "mean"
    })
)

display(
    svm_summary.head()
)

,Topics,Accuracy,BalancedAccuracy,Precision,Recall,F1,MCC
0,2,0.313333,0.313333,0.208858,0.313333,0.208413,0.262573
1,3,0.453333,0.453333,0.463720,0.453333,0.405031,0.407914
2,4,0.546667,0.546667,0.554821,0.546667,0.512556,0.506521
3,5,0.583333,0.583333,0.602726,0.583333,0.552512,0.545708
4,6,0.640000,0.640000,0.682528,0.640000,0.612366,0.611535


In [40]:
# ============================================================
# 5E-7 Standard Deviation Across Folds
# ============================================================

svm_std = (
    svm_results
    .groupby(
        "Topics",
        as_index=False
    )
    .agg({
        "Accuracy": "std",
        "BalancedAccuracy": "std",
        "Precision": "std",
        "Recall": "std",
        "F1": "std",
        "MCC": "std"
    })
)

svm_std = svm_std.rename(
    columns={
        "Accuracy": "AccuracyStd",
        "BalancedAccuracy": "BalancedAccuracyStd",
        "Precision": "PrecisionStd",
        "Recall": "RecallStd",
        "F1": "F1Std",
        "MCC": "MCCStd"
    }
)

display(
    svm_std.head()
)

,Topics,AccuracyStd,BalancedAccuracyStd,PrecisionStd,RecallStd,F1Std,MCCStd
0,2,0.027386,0.027386,0.053979,0.027386,0.035442,0.030343
1,3,0.043141,0.043141,0.067905,0.043141,0.037276,0.050955
2,4,0.054518,0.054518,0.035066,0.054518,0.055653,0.058103
3,5,0.042492,0.042492,0.079595,0.042492,0.046260,0.048667
4,6,0.045031,0.045031,0.065675,0.045031,0.055410,0.047128


In [41]:
# ============================================================
# 5E-8 Combined SVM Summary
# ============================================================

svm_summary = svm_summary.merge(
    svm_std,
    on="Topics",
    how="left"
)

display(
    svm_summary.head()
)

,Topics,Accuracy,BalancedAccuracy,Precision,Recall,F1,MCC,AccuracyStd,BalancedAccuracyStd,PrecisionStd,RecallStd,F1Std,MCCStd
0,2,0.313333,0.313333,0.208858,0.313333,0.208413,0.262573,0.027386,0.027386,0.053979,0.027386,0.035442,0.030343
1,3,0.453333,0.453333,0.463720,0.453333,0.405031,0.407914,0.043141,0.043141,0.067905,0.043141,0.037276,0.050955
2,4,0.546667,0.546667,0.554821,0.546667,0.512556,0.506521,0.054518,0.054518,0.035066,0.054518,0.055653,0.058103
3,5,0.583333,0.583333,0.602726,0.583333,0.552512,0.545708,0.042492,0.042492,0.079595,0.042492,0.046260,0.048667
4,6,0.640000,0.640000,0.682528,0.640000,0.612366,0.611535,0.045031,0.045031,0.065675,0.045031,0.055410,0.047128


In [42]:
# ============================================================
# 5E-9 Save SVM Summary
# ============================================================

svm_summary_file = (
    CLASSIFICATION_DIR /
    "svm_summary.csv"
)

svm_summary.to_csv(
    svm_summary_file,
    index=False
)

print(
    f"Saved:\n{svm_summary_file}"
)

Saved:
/content/drive/MyDrive/TopicEvalBench/classification/svm_summary.csv
